In [ ]:
import json
import pandas as pd

from pathlib import Path
from collections import Counter

from datasets import load_dataset, load_from_disk, ClassLabel

In [ ]:
UNKNOWN_CLASS_LABEL = "unk"

# ISIC 2020

In [ ]:
ISIC2020_DATASET_PATH = Path("/home/sulcm/datasets/isic2020/isic2020_builder/train")

In [ ]:
isic2020_gt = pd.read_csv(ISIC2020_DATASET_PATH / "ISIC_2020_Training_GroundTruth.csv")
isic2020_gt

In [ ]:
isic2020_gt["benign_malignant"].value_counts() / len(isic2020_gt)

In [ ]:
isic2020_gt["diagnosis"].value_counts() / len(isic2020_gt)

# ISIC 2019

In [ ]:
ISIC2019_DATASET_PATH = Path("/home/sulcm/datasets/isic2019")
COMMON_LABEL_MAPPING = { # isic2019 -> common labels
    "nv": "nv",
    "mel": "mel",
    "bkl": "bkl",
    "df": "df",
    "scc": "sccka",
    "bcc": "bcc",
    "vasc": "vasc",
    "ak": "akiec",
}

### Train

In [ ]:
isic2019_gt_one_hot = pd.read_csv(ISIC2019_DATASET_PATH / "isic2019_builder" / "ISIC_2019_Training_GroundTruth.csv").set_index("image", append=True)
for column in isic2019_gt_one_hot.columns:
    isic2019_gt_one_hot[column] = isic2019_gt_one_hot[column].astype(int)
labels = [l.lower() for l in isic2019_gt_one_hot.columns.to_list()]
isic2019_gt_classes = isic2019_gt_one_hot.dot(isic2019_gt_one_hot.columns).apply(lambda x: x.lower())

In [ ]:
isic2019_training_input = pd.read_csv(ISIC2019_DATASET_PATH / "isic2019_builder" / "ISIC_2019_Training_Metadata.csv")
isic2019_training_input

In [ ]:
isic2019_training_input[["file_name", "label"]] = isic2019_training_input.apply(
    lambda row: [
        row["image"] + ".jpg",
        COMMON_LABEL_MAPPING[isic2019_gt_classes.xs(row["image"], level=1).iloc[0]]
    ],
    axis=1, result_type="expand"
)

ds_info = {
    "labels": isic2019_training_input["label"].unique().tolist()
}

In [ ]:
isic2019_training_input.to_csv(ISIC2019_DATASET_PATH / "isic2019_builder" / "train" / "metadata.csv", index=False)
with open(ISIC2019_DATASET_PATH / "isic2019_builder" / "dataset_info.json", "w") as f:
    json.dump(ds_info, f, indent=2, ensure_ascii=False)

### Validation

In [ ]:
isic2019_val_gt_one_hot = pd.read_csv(ISIC2019_DATASET_PATH / "isic2019_builder" / "ISIC_2019_Test_GroundTruth.csv").set_index("image", append=True)
isic2019_val_gt_one_hot.drop(columns=["score_weight", "validation_weight"], inplace=True)
for column in isic2019_val_gt_one_hot.columns:
    isic2019_val_gt_one_hot[column] = isic2019_val_gt_one_hot[column].astype(int)
labels = [l.lower() for l in isic2019_val_gt_one_hot.columns.to_list()]
isic2019_gt_classes = isic2019_val_gt_one_hot.dot(isic2019_val_gt_one_hot.columns).apply(lambda x: x.lower())

In [ ]:
isic2019_val_input = pd.read_csv(ISIC2019_DATASET_PATH / "isic2019_builder" / "ISIC_2019_Test_Metadata.csv")
isic2019_val_input

In [ ]:
isic2019_val_input[["file_name", "label", "lesion_id"]] = isic2019_val_input.apply(
    lambda row: [
        row["image"] + ".jpg",
        COMMON_LABEL_MAPPING.get(isic2019_gt_classes.xs(row["image"], level=1).iloc[0], UNKNOWN_CLASS_LABEL),
        "UNK_" + row["image"]
    ],
    axis=1, result_type="expand"
)


len_with_unk_labels = len(isic2019_val_input)
isic2019_val_input = isic2019_val_input.loc[isic2019_val_input["label"] != UNKNOWN_CLASS_LABEL]
print(f"Valid labels for {len(isic2019_val_input)}/{len_with_unk_labels}")
isic2019_val_input

In [ ]:
isic2019_val_input.to_csv(ISIC2019_DATASET_PATH / "isic2019_builder" / "validation" / "metadata.csv", index=False)

In [ ]:
dataset = load_dataset("imagefolder", data_dir=ISIC2019_DATASET_PATH / "isic2019_builder")
dataset

In [ ]:
dataset = dataset.cast_column("label", ClassLabel(names=list(COMMON_LABEL_MAPPING.values())))

In [ ]:
# dataset.save_to_disk(ISIC2019_DATASET_PATH / "isic2019")

## Use build ISIC2019 dataset

In [ ]:
lds = load_from_disk(
    dataset_path=ISIC2019_DATASET_PATH / "isic2019"
)
lds

In [ ]:
# lds.cleanup_cache_files()

In [ ]:
Counter(lds["train"]["age_approx"])

# HAM10000

In [ ]:
import json
import matplotlib.pyplot as plt

In [ ]:
ham_slice = isic2019_training_input.apply(lambda row: row if isinstance(row["lesion_id"], str) and row["lesion_id"].startswith("HAM") else None, axis=1).dropna()

In [ ]:
with open("./ham_ids_from_isic2019.json", "w") as f:
    json.dump(ham_slice.index.tolist(), f)

In [ ]:
HAM_DATASET_PATH = Path("/home/sulcm/datasets/ham10000")
COMMON_LABEL_MAPPING = { # isic2019 -> common labels
    "nv": "nv",
    "mel": "mel",
    "bkl": "bkl",
    "df": "df",
    "scc": "sccka",
    "bcc": "bcc",
    "vasc": "vasc",
    "ak": "akiec",
}

In [ ]:
ham_gt_one_hot = pd.read_csv(HAM_DATASET_PATH / "ISIC2018_Task3_Training_GroundTruth" / "ISIC2018_Task3_Training_GroundTruth.csv").set_index("image", append=True)
for column in ham_gt_one_hot.columns:
    ham_gt_one_hot[column] = ham_gt_one_hot[column].astype(int)
labels = [l.lower() for l in ham_gt_one_hot.columns.to_list()]
ham_gt_classes = ham_gt_one_hot.dot(ham_gt_one_hot.columns).apply(lambda x: x.lower())

In [ ]:
ham_gt_classes

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(1*5, 3))
s_counts = Counter(ham_gt_classes)
sorted_labels = [l for l, c in s_counts.most_common()]
for i, s in enumerate(["train"]):
    _ax = ax
    rects = _ax.bar(
        list(range(len(s_counts))),
        [s_counts.get(l, 0.0) for l in sorted_labels],
        label=sorted_labels
    )
    _ax.bar_label(rects, padding=1)
    _ax.set_title(f"{s.capitalize()} split")
    _ax.set_xticks(list(range(len(sorted_labels))), sorted_labels, rotation=45)
    _ax.set_ylim(0, max(s_counts.values()) * 1.1)
    if i == 0:
        _ax.set_ylabel("Number of samples")
plt.tight_layout()
plt.savefig("/home/sulcm/school/ST/DP/images/ham10000_histogram.pdf", bbox_inches="tight")
plt.show()

In [ ]:
ham_ids = [idx for idx, l_id in enumerate(lds["train"]["lesion_id"]) if l_id is not None and l_id.lower().startswith("ham")]

In [ ]:
lds["train"][0]["image"].size

In [ ]:
ham_shapes = [lds["train"][l_id]["image"].size for l_id in ham_ids]

In [ ]:
ham_shapes

In [ ]:
ham_labels = [lds["train"][l_id]["label"] for l_id in ham_ids]

In [ ]:
Counter(lds["train"]["label"][ham_ids])

In [ ]:
lds["train"].features